In [2]:
import pandas as pd
import numpy as np

In [3]:
hh_test     = pd.read_csv('test_households.csv')
pers_test   = pd.read_csv('test_persons.csv')
trips_test  = pd.read_csv('test_trips.csv')
hh_train    = pd.read_csv('train_val_households.csv')
pers_train  = pd.read_csv('train_val_persons.csv')
trips_train = pd.read_csv('train_val_trips.csv')

print("Files loaded")
print("trips_train shape:", trips_train.shape)
print("trips_test shape: ", trips_test.shape)

Files loaded
trips_train shape: (13641, 16)
trips_test shape:  (6223, 15)


In [4]:
# Step 1: trips + persons (match on both hhid AND persid)
train = trips_train.merge(pers_train, on=['hhid', 'persid'], how='left')
test  = trips_test.merge(pers_test,   on=['hhid', 'persid'], how='left')

# Step 2: result + households (match on hhid)
# suffixes handles the duplicate 'travdow' column
train = train.merge(hh_train, on='hhid', how='left', suffixes=('', '_hh'))
test  = test.merge(hh_test,   on='hhid', how='left', suffixes=('', '_hh'))

# Step 3: drop duplicate travdow (identical in both persons and households)
train = train.drop(columns=['travdow_hh'])
test  = test.drop(columns=['travdow_hh'])

# Sanity checks
assert len(train) == len(trips_train), "Rows lost in train merge!"
assert len(test)  == len(trips_test),  "Rows lost in test merge!"

print("Tables joined")
print("Train shape:", train.shape)  # expect (13641, 61)
print("Test shape: ", test.shape)   # expect (6223, 60)

Tables joined
Train shape: (13641, 61)
Test shape:  (6223, 60)


In [6]:
# Check what's missing
print("=== Missing Values ===")
missing = train.isnull().sum()
print(missing[missing > 0])

# Fix cumdist: fill with median from TRAINING data only
median_dist = train['cumdist'].median()
train['cumdist'] = train['cumdist'].fillna(median_dist)
test['cumdist']  = test['cumdist'].fillna(median_dist)
print(f"\ncumdist NaNs filled with median: {median_dist:.5f} km")

# Fix hhinc_group: fill with 'Unknown' category
train['hhinc_group'] = train['hhinc_group'].fillna('Unknown')
test['hhinc_group']  = test['hhinc_group'].fillna('Unknown')
print("hhinc_group NaNs filled with 'Unknown'")

# Verify no missing values remain
print("\nMissing values remaining:")
remaining = train.isnull().sum()
print(remaining[remaining > 0])
print("(empty = no missing values)")

=== Missing Values ===
Series([], dtype: int64)

cumdist NaNs filled with median: 3.75347 km
hhinc_group NaNs filled with 'Unknown'

Missing values remaining:
Series([], dtype: int64)
(empty = no missing values)


In [7]:
# Check who is marked 'Not applicable' for anywork
not_applicable = train[train['anywork'] == 'Not applicable']
print(f"'Not applicable' for anywork: {len(not_applicable)} people")
print("\nWhat is their main activity?")
print(not_applicable['mainact'].value_counts())
# Expected: all children/students — confirming the data is clean

'Not applicable' for anywork: 2109 people

What is their main activity?
mainact
Primary School       1187
Not Yet at School     565
Secondary School      357
Name: count, dtype: int64


In [8]:
# Calculate speed (km/h) for each trip
# clip(lower=1) prevents division by zero
train['cumdist']  = pd.to_numeric(train['cumdist'],  errors='coerce')
train['travtime'] = pd.to_numeric(train['travtime'], errors='coerce')
train['speed_kmh'] = (train['cumdist'] / train['travtime'].clip(lower=1)) * 60

# Flag implausible speeds per mode
walk_flag  = (train['mode'] == 'WALK')                       & (train['speed_kmh'] > 15)
cycle_flag = (train['mode'] == 'CYCLE')                      & (train['speed_kmh'] > 60)
drive_flag = (train['mode'].isin(['DRIVE', 'PASSENGER']))    & (train['speed_kmh'] > 200)

train['flag_speed'] = (walk_flag | cycle_flag | drive_flag).astype(int)

# Flag WFH without employment
wfh_flag = (train['wfhtravday'] == 'Yes') & (train['anywork'] != 'Yes')
train['flag_wfh_no_job'] = wfh_flag.astype(int)

# Combined flag
train['flag_any'] = ((train['flag_speed'] == 1) |
                     (train['flag_wfh_no_job'] == 1)).astype(int)

# Report
print("=== Quality Check Results ===")
print(f"WALK  > 15 km/h:       {walk_flag.sum()} trips")
print(f"CYCLE > 60 km/h:       {cycle_flag.sum()} trips")
print(f"DRIVE > 200 km/h:      {drive_flag.sum()} trips")
print(f"WFH without job:       {wfh_flag.sum()} trips")
print(f"\nTotal flagged:         {train['flag_any'].sum()} ({train['flag_any'].mean():.1%} of data)")

=== Quality Check Results ===
WALK  > 15 km/h:       18 trips
CYCLE > 60 km/h:       2 trips
DRIVE > 200 km/h:      2 trips
WFH without job:       0 trips

Total flagged:         22 (0.2% of data)


In [10]:
# Save clean tables 
train.to_csv('clean_train.csv', index=False)
test.to_csv('clean_test.csv',   index=False)

print(f"clean_train.csv saved — shape: {train.shape}")
print(f"clean_test.csv saved  — shape: {test.shape}")

clean_train.csv saved — shape: (13641, 65)
clean_test.csv saved  — shape: (6223, 60)
